### Notebook for exploring Self-Conditioned Knots for Probabilistic LiDAR Hazrad Fields

In [1]:
from vedo import *
settings.default_backend = "k3d"
import numpy as np
from scipy.spatial.transform import Rotation as R
import copy

from matplotlib import pyplot as p

%load_ext autoreload
%autoreload 2
%autosave 180

Autosaving every 180 seconds


In [2]:
# Configure device (GPU if available)
import torch
from torch.utils.data import Dataset, DataLoader

# Reset default device to CPU to avoid DataLoader sampler conflicts
torch.set_default_device(None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")
print(f"Using device: {device}")


CUDA available: True
Device name: NVIDIA GeForce GTX 1650 with Max-Q Design
Device count: 1
Using device: cuda


## Generate synthetic LiDAR data

In [3]:
from synthetic_data_generation_utils import (
    LidarProjectionModel,
    Scene,
    generate_point_cloud,
)

# 1. LiDAR Intrinsics (e.g. 32-channel, 360-degree rotation)
sensor = LidarProjectionModel(
    elevation_min=-25.0,
    elevation_max=15.0,
    num_detectors=32,
    num_azimuth=360,
    max_range=30.0,
)

# 2. Build 10x10m Enclosed Scene
scene = Scene()
wall_density = 100.0  # Solid boundaries (near-surface returns)

# --- Floor & 4 Walls (10m x 10m x 3m interior) ---
scene.add_box(center=[0.0, 0.0, -0.1], size=[10.4, 10.4, 0.2], density=wall_density, name="floor")
scene.add_box(center=[0.0, 5.1, 1.5],  size=[10.4, 0.2, 3.0],  density=wall_density, name="wall_north")
scene.add_box(center=[0.0, -5.1, 1.5], size=[10.4, 0.2, 3.0],  density=wall_density, name="wall_south")
scene.add_box(center=[5.1, 0.0, 1.5],  size=[0.2, 10.4, 3.0],  density=wall_density, name="wall_east")
scene.add_box(center=[-5.1, 0.0, 1.5], size=[0.2, 10.4, 3.0],  density=wall_density, name="wall_west")

# --- 4 Cylinders (Pillars) with density = 1.0 ---
cyl_density = 1.0
scene.add_cylinder(center=[2.5, 2.5, 1.5],   radius=0.4, height=3.0, density=cyl_density, name="pillar_NE")
scene.add_cylinder(center=[-2.5, 2.5, 1.5],  radius=0.4, height=3.0, density=cyl_density, name="pillar_NW")
scene.add_cylinder(center=[2.5, -2.5, 1.5],  radius=0.4, height=3.0, density=cyl_density, name="pillar_SE")
scene.add_cylinder(center=[-2.5, -2.5, 1.5], radius=0.4, height=3.0, density=cyl_density, name="pillar_SW")

# --- Low-Density Spheres (Porous foliage / clutter with deep volumetric penetration) ---
scene.add_sphere(center=[2.0, 0.0, 1.0],   radius=0.8,  density=0.15, name="sphere_1")
scene.add_sphere(center=[-2.0, 0.5, 1.2],  radius=0.9,  density=0.10, name="sphere_2")
scene.add_sphere(center=[0.0, 3.0, 1.5],   radius=1.1,  density=0.08, name="sphere_3")
scene.add_sphere(center=[0.0, -2.8, 1.0],  radius=0.7,  density=0.20, name="sphere_4")
scene.add_sphere(center=[3.2, -1.8, 1.3],  radius=1.0,  density=0.12, name="sphere_5")
scene.add_sphere(center=[-3.0, -2.0, 1.4], radius=0.85, density=0.18, name="sphere_6")

# 3. Generate Point Cloud with the sensor positioned at the center of the room [0, 0, 1.5]
sensor_pos = [0.0, 0.0, 1.5]
point_cloud = generate_point_cloud(
    sensor=sensor,
    scene=scene,
    sensor_position=sensor_pos,
    method="sample",             # Stochastic sampling based on return probabilities
    return_placement="volume",   # Penetrable spheres have points distributed throughout their interior
    random_seed=42,
)

print(f"Generated {len(point_cloud)} points from {sensor.total_rays} rays.")
cloud_i = np.asarray(np.shape(point_cloud.points))
print(cloud_i)

# 4. Interactive 3D visualization with vedo
plt = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp = scene.to_vedo(alpha=0.25)
disp.append(point_cloud.to_vedo(color_by="probability", cmap="viridis", point_size=3.5))
plt.show(disp, "10x10m Enclosed Room with Pillars and Low-Density Spheres")


Generated 11337 points from 11520 rays.
[11337     3]


## generate training data from multiple viewpoints

In [4]:
import numpy as np
from synthetic_data_generation_utils import compute_look_at_rotation

def generate_orbit_poses(num_poses: int = 16, radius: float = 3.0, height: float = 1.5, target=[0.0, 0.0, 1.2]):
    """Generates N sensor poses in a circular ring facing the center."""
    angles = np.linspace(0, 2 * np.pi, num_poses, endpoint=False)
    positions = []
    rotations = []
    
    for theta in angles:
        # Sensor position along the circle
        pos = np.array([radius * np.cos(theta), radius * np.sin(theta), height])
        # Aim the sensor forward (+X in sensor local frame) toward the target point
        rot = compute_look_at_rotation(eye=pos, target=target)
        positions.append(pos)
        rotations.append(rot)
        
    return np.array(positions), np.array(rotations)


In [5]:
from synthetic_data_generation_utils import (
    LidarProjectionModel, 
    Scene, 
    trace_scene_rays,
    rays_to_plucker,
)

# 1. Setup Intrinsics & Scene
sensor = LidarProjectionModel(elevation_min=-25.0, elevation_max=15.0, num_detectors=32, num_azimuth=180)

# 2. Get N Poses
N_poses = 12 #12
positions, rotations = generate_orbit_poses(num_poses=N_poses, radius=1., height=1.4)

# 3. Collect Ray Data from all viewpoints (Zero test-point sampling!)
all_rays_o = []
all_rays_d = []
all_hit_ranges = []
all_hit_masks = []

for i in range(N_poses):
    print(i)
    pos, rot = positions[i], rotations[i]
    rays_o, rays_d = sensor.get_rays(pos, rot, flatten=True)
    
    # Pure geometric ray tracing: returns hit range and hit mask directly
    trace_results = trace_scene_rays(rays_o, rays_d, scene, max_range=sensor.max_range)
    
    for r_idx, res in enumerate(trace_results):
        hits = res["hits"]
        if len(hits) > 0:
            hit_range = hits[0]["t_in"]
            hit_mask = 1.0
        else:
            hit_range = sensor.max_range
            hit_mask = 0.0
            
        all_rays_o.append(rays_o[r_idx])
        all_rays_d.append(rays_d[r_idx])
        all_hit_ranges.append(hit_range)
        all_hit_masks.append(hit_mask)

# Stack into consolidated arrays
rays_o = np.array(all_rays_o, dtype=np.float32)          # (N_total_rays, 3)
rays_d = np.array(all_rays_d, dtype=np.float32)          # (N_total_rays, 3)
hit_ranges = np.array(all_hit_ranges, dtype=np.float32)  # (N_total_rays,)
hit_masks = np.array(all_hit_masks, dtype=np.float32)    # (N_total_rays,)


0
1
2
3
4
5
6
7
8
9
10
11


In [8]:
import numpy as np
from vedo import Plotter, Points
from synthetic_data_generation_utils import PointCloud

# 1. Reproject ray origins + directions out to 3D Cartesian points
valid_hits = (hit_masks > 0.0)
pts_3d = rays_o[valid_hits] + hit_ranges[valid_hits, np.newaxis] * rays_d[valid_hits]
ranges_3d = hit_ranges[valid_hits]

print(f"Reprojected {len(pts_3d):,} 3D points from {len(hit_masks):,} rays "
      f"({valid_hits.mean() * 100:.1f}% hit rate).")

# 2. Package into PointCloud container
cloud_train = PointCloud(
    points=pts_3d,
    probabilities=hit_masks[valid_hits],
    ranges=ranges_3d,
)

# 3. Interactive 3D visualization overlaid with scene geometry
plt_vedo = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp = scene.to_vedo(alpha=0.15) if 'scene' in locals() else []

# Add training point cloud (colored by distance from sensor)
cloud_actor = cloud_train.to_vedo(color_by='range', cmap='plasma', point_size=3.0)
disp.append(cloud_actor)

# Add red markers for sensor orbit positions
if 'positions' in locals():
    disp.append(Points(positions, r=8.0, c='red'))

plt_vedo.show(disp, f"Multi-View Training Data ({len(pts_3d):,} Points from {N_poses} Poses)")


Reprojected 66,843 3D points from 69,120 rays (96.7% hit rate).


In [9]:
# Structure into torch dataset for training
from sck_model import LidarRayDataset, GPURayBatcher

# Define the network

In [10]:
import torch
import torch.nn as nn
from sck_model import PositionalEncoding, DeeperSCKNetwork, SCKNetwork

In [11]:
from sck_model import exact_continuous_cdf_loss, interpolate_cdf_to_grid, compute_sck_loss, SCKLoss


# Train Loop

### Notes: Self-Conditioned Residual Knot Flow

In true Self-Conditioning:
1. **Initial State (Round 0)**: Knots start cleanly and uniformly distributed across $[s_{\min}, s_{\max}]$ ($[0.1\text{m}, 15.0\text{m}]$).
2. **Residual Knot Flow**: Rather than re-generating absolute knot positions via an unconstrained global softmax from scratch every pass, the network outputs a local displacement field $\delta s_k = \tanh(\text{out}_0) \times d_{\max}$ that physically nudges candidate knots toward high-hazard / surface boundaries ($s^{(r+1)} = \text{sort}(s^{(r)} + \delta s)$).
3. **Self-Conditioning Refinement**: Subsequent rounds evaluate features at the updated knot positions, allowing knots to bracket sharp jumps tightly while empty regions stay naturally uniform.


In [17]:
import torch
import logging
from sck_model import train_step as sck_train_step, train_epoch, train_model

# Suppress benign Inductor SM-count warning on mobile GPUs
logging.getLogger("torch._inductor.utils").setLevel(logging.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeeperSCKNetwork(
    num_knots=32,
    num_rounds=4,
    max_displacement=1.0,
    hidden_dim=256,
    num_hidden_blocks=2,
).to(device)

# Compile model for CUDA kernel fusion (~1.8x speedup on forward pass)
compiled_model = torch.compile(model)
optimizer = torch.optim.Adam(compiled_model.parameters(), lr=1e-4)

# Optional: dense grid used strictly for high-resolution plotting/visualization
eval_grid = torch.linspace(0.1, 15.0, 256, device=device)

def train_step(batch):
    # Grid-free exact analytic continuous loss (no eval_grid required!)
    return sck_train_step(compiled_model, optimizer, batch, device=device)


Total model parameters: 657,667


In [ ]:
import matplotlib.pyplot as plt
import math

# 1. Instantiate Fast GPU Batcher (pure grid-free using exact hit_ranges and hit_masks)
dataset = LidarRayDataset(
    rays_o=rays_o,
    rays_d=rays_d,
    hit_ranges=hit_ranges,
    hit_masks=hit_masks,
)
dataloader = GPURayBatcher(
    rays_o=rays_o,
    rays_d=rays_d,
    hit_ranges=hit_ranges,
    hit_masks=hit_masks,
    batch_size=128,
    shuffle=True,
    device=device,
)

# 2. Run Training Loop with Exact Grid-Free Analytic Loss & PLiNK Ray-Drop Head
num_epochs = 20
history = train_model(
    model=compiled_model,
    dataloader=dataloader,
    optimizer=optimizer,
    eval_grid=None,            # None = Exact closed-form grid-free analytic loss!
    num_epochs=num_epochs,
    device=device,
    augment_origin_slide=True, # Vectorized GPU origin sliding on hit_ranges
    l1_weight=0.5,             # Wasserstein-1 Earth Mover's Distance
    l2_weight=1.0,             # Continuous Ranked Probability Score (CRPS)
    lambda_raydrop=0.1,        # PLiNK ray-drop BCE loss
)

# 3. Visualize Learned Knot Placements vs Analytic Ground Truth Step Function
model.eval()
with torch.no_grad():
    sample_indices = [0, 50, 150, 300]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for ax, idx in zip(axes.flat, sample_indices):
        sample = dataset[idx]
        o = sample['ray_o'].unsqueeze(0).to(device)
        d = sample['ray_d'].unsqueeze(0).to(device)
        s_knots, c_knots, _, p_hit = model(o, d)
        s_k = s_knots[0].cpu().numpy()
        c_k = c_knots[0].cpu().numpy()

        # Dense interpolation purely for rendering the predicted continuous curve
        pred_interp = interpolate_cdf_to_grid(s_knots, c_knots, eval_grid)[0].cpu().numpy()
        eval_pts = eval_grid.cpu().numpy()

        # Analytic ground truth step function (PLiNK style)
        if sample['hit_mask'] > 0:
            r_hit = sample['hit_range'].item()
            ax.step([0.1, r_hit, r_hit, 15.0], [0.0, 0.0, 1.0, 1.0], 'k--', label=f'GT Step ({r_hit:.2f}m)', linewidth=2, where='post')
            ax.axvline(r_hit, color='green', linestyle=':', alpha=0.6)
        else:
            ax.axhline(0.0, color='gray', linestyle='--', label='GT Sky (Non-Return)', linewidth=2)
            ax.set_title(f'Non-Return Ray (p_hit={p_hit.item():.2f})', fontsize=9, color='purple')

        ax.plot(eval_pts, pred_interp, 'b-', label='Predicted Continuous Spline')
        ax.scatter(s_k, c_k, color='red', s=25, zorder=5, label='Learned Knots')
        ax.set_xlim(0.0, 15.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('Range (m)')
        ax.set_ylabel('CDF')
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


Training on cuda (69120 rays, 540 batches/epoch) [Exact Grid-Free Analytic Loss]...
Epoch [01/20] - Loss: 4.342226
Epoch [02/20] - Loss: 1.279093
Epoch [03/20] - Loss: 0.872284
Epoch [04/20] - Loss: 0.747351
Epoch [05/20] - Loss: 0.725953
Epoch [06/20] - Loss: 0.726277
Epoch [07/20] - Loss: 0.637333
Epoch [08/20] - Loss: 0.603339


# Make cloud at novel position

In [ ]:
import torch
import numpy as np
from synthetic_data_generation_utils import PointCloud
from vedo import Plotter
from sck_model import render_novel_point_cloud, generate_point_cloud_from_network

# ==============================================================================
# Execute Inference with PLiNK Ray-Drop Filtering & Visualize
# ==============================================================================

# 1. Generate point cloud with the sensor placed at room center [0, 0, 1.5]
#    Rays with P(non-return) > non_return_threshold are pruned to suppress sky floaters.
pred_cloud = render_novel_point_cloud(
    model=model,
    sensor=sensor,
    sensor_position=[0.0, 0.0, 1.5],
    method= 'threshold', #'sample',             # Options: 'sample' (stochastic) or 'threshold' (median/quantile)
    threshold=0.7,               # CDF threshold if using method='threshold' (0.5 = median return)
    non_return_threshold=0.6,    # Prune ray if likelihood of non-return > 50%
    device=device,
    verbose=True,                # Prints total rays, returned points, and dropped rays
)

# 2. 3D interactive visualization with Vedo (overlaid on scene geometry)
plt_vedo = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp = scene.to_vedo(alpha=0.15)
disp.append(pred_cloud.to_vedo(color_by='range', cmap='plasma', point_size=3.5))
plt_vedo.show(disp, 'Network Reconstructed Point Cloud (with Ray-Drop Filtering)')


In [ ]:
print("Model device:", next(model.parameters()).device)
print(f"Allocated VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")